# Sales Forecasting
### Retail / E-commerce Time-Series Project

**Goal:** Predict future sales from historical sales data while considering trend, seasonality and promotional factors.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pathlib import Path

DATA = Path("../data/sales_data.csv")
OUT = Path("../outputs")
OUT.mkdir(exist_ok=True)
df = pd.read_csv(DATA, parse_dates=["date"]).set_index("date").asfreq("MS")
df.head()


## 1. Exploratory Data Analysis
We inspect the trend, seasonality and promotional periods.

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(df.index, df.sales)
plt.title("Historical Monthly Sales")
plt.xlabel("Date"); plt.ylabel("Sales")
plt.grid(alpha=.25); plt.show()

monthly = df.groupby(df.index.month).sales.mean()
plt.figure(figsize=(10,4))
plt.plot(monthly.index, monthly.values, marker="o")
plt.title("Average Sales by Month")
plt.xlabel("Month"); plt.ylabel("Average Sales")
plt.xticks(range(1,13)); plt.grid(alpha=.25); plt.show()


In [ ]:
promo_summary = df.groupby("promotion")["sales"].mean()
print("Average sales without promotion:", round(promo_summary.loc[0],2))
print("Average sales with promotion:", round(promo_summary.loc[1],2))
promo_summary


## 2. Train/Test Split
The final 12 months are kept as an unseen test set.

In [ ]:
test_size = 12
train = df.iloc[:-test_size].copy()
test = df.iloc[-test_size:].copy()

features = ["promotion","holiday_season","discount_pct","ad_spend"]
print(train.shape, test.shape)


## 3. SARIMAX Forecasting
SARIMAX extends ARIMA by allowing seasonal structure and external variables such as promotions and advertising spend.

In [ ]:
model = SARIMAX(
    train["sales"],
    exog=train[features],
    order=(1,1,1),
    seasonal_order=(1,1,1,12),
    enforce_stationarity=False,
    enforce_invertibility=False
)
fit = model.fit(disp=False)
print(fit.summary())


In [ ]:
pred = fit.get_forecast(steps=len(test), exog=test[features]).predicted_mean
mae = mean_absolute_error(test.sales, pred)
rmse = mean_squared_error(test.sales, pred) ** 0.5
print(f"MAE: {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")


In [ ]:
plt.figure(figsize=(12,5))
plt.plot(train.index, train.sales, label="Train")
plt.plot(test.index, test.sales, label="Actual")
plt.plot(test.index, pred, label="Forecast")
plt.axvline(test.index[0], linestyle="--", alpha=.6)
plt.title("Actual vs Forecast")
plt.xlabel("Date"); plt.ylabel("Sales")
plt.grid(alpha=.25); plt.legend(); plt.show()


## 4. Future 12-Month Forecast
For demonstration, future promotional/holiday flags and planned advertising assumptions are supplied as model inputs.

In [ ]:
future_dates = pd.date_range(df.index[-1] + pd.offsets.MonthBegin(1), periods=12, freq="MS")
future = pd.DataFrame(index=future_dates)
future["promotion"] = future.index.month.isin([8,11,12]).astype(int)
future["holiday_season"] = future.index.month.isin([10,11,12]).astype(int)
future["discount_pct"] = np.where(future.promotion==1, 20, 5)
future["ad_spend"] = np.where(future.promotion==1, 23000, 12000)

fc = fit.get_forecast(12, exog=future[features])
future["forecast_sales"] = fc.predicted_mean.values
ci = fc.conf_int()
future["lower_95"] = ci.iloc[:,0].values
future["upper_95"] = ci.iloc[:,1].values
future.round(2)


In [ ]:
plt.figure(figsize=(12,5))
plt.plot(df.index, df.sales, label="Historical sales")
plt.plot(future.index, future.forecast_sales, label="Future forecast")
plt.fill_between(future.index, future.lower_95, future.upper_95, alpha=.2, label="95% interval")
plt.title("Next 12 Months Sales Forecast")
plt.xlabel("Date"); plt.ylabel("Sales")
plt.grid(alpha=.25); plt.legend(); plt.show()


## 5. Business Interpretation
- Trend shows the long-term direction of sales.
- Monthly seasonality captures recurring patterns.
- Promotion and holiday indicators allow the model to account for campaign periods.
- Advertising spend is included as an external regressor.
- MAE and RMSE quantify forecast error on the unseen test period.

**Conclusion:** The project demonstrates an end-to-end sales forecasting workflow from raw time-series data to evaluation and future forecasting.